# Replication on Colab (GPU)

Runs the TwitterAAE replication with a free GPU. First: **Runtime → Change runtime type → T4 GPU**.

Files are kept on your Google Drive so a disconnect does not lose the download or the results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

Clone your GitHub copy of the project (replace the URL with yours). If the repo is private, use a personal access token or make it public for the run.

In [ ]:
REPO = 'https://github.com/YOUR-USERNAME/demog-leakage.git'
!git clone -q $REPO /content/demog-leakage || (cd /content/demog-leakage && git pull -q)
%cd /content/demog-leakage
!pip -q install -r requirements.txt

Keep raw data, models and results on Drive (created once, reused on later sessions).

In [ ]:
import os
D = '/content/drive/MyDrive/comp8240'
for sub in ['raw', 'processed', 'models', 'results']:
    os.makedirs(f'{D}/{sub}', exist_ok=True)
!rm -rf data/raw data/processed models results
!ln -s $D/raw data/raw && ln -s $D/processed data/processed && ln -s $D/models models && ln -s $D/results results
!ls -la data models results

## 1. Download TwitterAAE (skipped if already on Drive)

In [ ]:
import os
if not os.path.isfile('data/raw/TwitterAAE-full-v1.zip'):
    !bash scripts/get_twitteraae.sh
else:
    !ls -lh data/raw/TwitterAAE-full-v1.zip

## 2. Build the balanced sentiment/race data

In [ ]:
!python src/prepare_twitteraae.py data/raw/TwitterAAE-full-v1.zip data/processed/sent_race
# optional, for the investigation: the original code's behaviour, quirks included
# !python src/prepare_twitteraae.py data/raw/TwitterAAE-full-v1.zip data/processed/sent_race_faithful --mode faithful

## 3. Run the four experiments (roughly 1-2 hours on a T4 GPU)

In [ ]:
!python src/experiments.py run-all --data data/processed/sent_race --epochs 20 --tag twitteraae

In [ ]:
print(open('results/twitteraae/results.md').read())

Results and logs are now in `MyDrive/comp8240/results/twitteraae/`. Copy that folder into the repo's `results/` and commit it.